# Local WSL: Qwen3-4B Vietnamese GEC QLoRA

Select the **HiGANplus WSL Python kernel**. This notebook runs the **current local checkout**, including unpushed changes; it does not clone GitHub or use Kaggle paths/secrets.

Defaults: full balanced dataset, seed 42, five epochs, one CUDA GPU. Other settings come from `configs/baseline.json`. The training cell streams stdout/stderr here and saves the same output to a `.log` file beside the run directory. Model loading and epoch validation can have quiet periods.

**Dependency setup installs the HiGANplus-matched versions in `requirements.txt`, retaining CUDA PyTorch 2.7.1.** The existing Hugging Face stack already matches; PEFT is the added training dependency. Set `INSTALL_DEPENDENCIES = False` only when these requirements are already installed.

Run cells in order. The preflight cell does not train. Only the clearly marked **Start training** cell launches the full experiment. Interrupt that cell to stop the entire launched process group; a cancelled run is partial, not resumable through this notebook. Rerun the configuration cell to get a fresh output directory.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import importlib.metadata
import json
import os
import signal
import subprocess
import sys
import tempfile

EXPECTED_ENV = "HiGANplus"
INSTALL_DEPENDENCIES = True
# Leave None when opening this notebook from the project root or notebooks/.
# Otherwise set the Linux project directory explicitly (not a Windows UNC path).
PROJECT_ROOT_OVERRIDE = None
CONFIG_PATH = "configs/baseline.json"
DATA_PATH = "dataset/newdataset/VGEC_balanced.json"
RUN_CONFIG_OVERRIDES = {"run": {"seeds": [42]}}
# Optional short CUDA smoke run instead of full training:
# RUN_CONFIG_OVERRIDES = {
#     "run": {"seeds": [42]},
#     "data": {"train_limit": 10, "validation_limit": 5},
#     "training": {"args": {"max_steps": 1, "num_train_epochs": 1, "warmup_steps": 0}},
# }

if sys.platform != "linux" or Path(sys.prefix).name != EXPECTED_ENV:
    raise RuntimeError(
        f"Select the {EXPECTED_ENV} WSL kernel. Current Python: {sys.executable} (prefix: {sys.prefix})"
    )
required_files = ["requirements.txt", "model/train.py", "tools/kaggle_run.py", CONFIG_PATH]
if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_DIR = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    PROJECT_DIR = next(
        (directory for directory in [Path.cwd(), *Path.cwd().parents]
         if all((directory / filename).is_file() for filename in required_files)),
        None,
    )
if PROJECT_DIR is None or not all((PROJECT_DIR / filename).is_file() for filename in required_files):
    raise FileNotFoundError("Open from the project root/notebooks, or set PROJECT_ROOT_OVERRIDE to the local checkout.")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
print("Kernel:", sys.executable)
print("Local checkout:", PROJECT_DIR)

In [ ]:
def run_streamed(command, *, log_path=None):
    """Stream child stdout/stderr into this cell; stop all descendants on interrupt/error."""
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    print("Command:", subprocess.list2cmdline([str(arg) for arg in command]), flush=True)
    with tempfile.TemporaryFile(mode="w+", encoding="utf-8") if log_path is None else Path(log_path).open("a", encoding="utf-8") as log:
        process = subprocess.Popen(
            command, cwd=PROJECT_DIR, env=environment,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
            start_new_session=True,
        )
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
            if returncode:
                raise subprocess.CalledProcessError(returncode, command)
        except BaseException:
            # A separate session contains launcher -> experiment -> training.
            # Killing just the launcher can leave GPU training running.
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                pass
            # Clean up descendants even if the group leader exited first.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
            print("\nLaunched process group stopped; any run artifacts are partial.", flush=True)
            raise
        finally:
            process.stdout.close()

## Dependency setup

Uses this kernel's Python, retains its existing CUDA PyTorch, and installs `requirements.txt`. If imported package versions change, restart the kernel and rerun from the first code cell. No model weights are loaded in the notebook kernel; training uses a subprocess so interruption can release its GPU memory.

In [ ]:
from packaging.version import Version

installed_torch = importlib.metadata.version("torch")
if Version(installed_torch).base_version != "2.7.1":
    raise RuntimeError(f"This project's HiGANplus stack requires CUDA PyTorch 2.7.1; found {installed_torch} in {EXPECTED_ENV}.")
packages = ["transformers", "peft", "accelerate", "bitsandbytes", "safetensors", "huggingface-hub", "tokenizers"]
imported_before = {
    name: importlib.metadata.version(name)
    for name in packages if name.replace("-", "_") in sys.modules
}
if INSTALL_DEPENDENCIES:
    with tempfile.TemporaryDirectory() as temporary:
        constraints = Path(temporary) / "constraints.txt"
        constraints.write_text(f"torch=={installed_torch}\n", encoding="utf-8")
        run_streamed([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt"),
                      "--constraint", str(constraints)])
    changed_imports = [name for name, old in imported_before.items() if importlib.metadata.version(name) != old]
    if changed_imports:
        raise RuntimeError(f"Restart the kernel to reload {changed_imports}, then rerun cells in order.")
run_streamed([sys.executable, "-c",
    "import torch, transformers, peft, accelerate, bitsandbytes; "
    "print('Training imports passed'); "
    "print('torch:', torch.__version__, 'CUDA:', torch.cuda.is_available()); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"])
print("Retained PyTorch:", installed_torch)

## Resolve a fresh run configuration

Nested overrides do not edit `configs/baseline.json`. Default local data is private and must already exist. Override `data.path` if using a different bundle. Configuration snapshots, outputs, and logs remain under the configured output root (normally `runs/`). Rerun this cell for each new run; existing nonempty run directories are rejected by the launcher.

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_config = json.loads((PROJECT_DIR / CONFIG_PATH).read_text(encoding="utf-8"))
run_config["data"]["path"] = DATA_PATH

def apply_overrides(target, overrides):
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            apply_overrides(target[key], value)
        else:
            target[key] = value

apply_overrides(run_config, RUN_CONFIG_OVERRIDES)
data_file = Path(run_config["data"]["path"]).expanduser()
if not data_file.is_absolute():
    data_file = PROJECT_DIR / data_file
data_file = data_file.resolve()
if not data_file.is_file():
    raise FileNotFoundError(f"Missing local balanced dataset: {data_file}. Build it with tools/build_dataset.py first.")
run_config["data"]["path"] = str(data_file)
output_root = Path(run_config["run"]["output_root"]).expanduser()
if not output_root.is_absolute():
    output_root = PROJECT_DIR / output_root
output_root = output_root.resolve()
config_root = output_root / ".local-configs"
config_root.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = output_root / f"{run_config['run']['name']}-local-{RUN_ID}"
LOG_PATH = Path(str(OUTPUT_DIR) + ".log")
config_file = config_root / f"{RUN_ID}.json"
config_file.write_text(json.dumps(run_config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
command = [sys.executable, str(PROJECT_DIR / "tools/kaggle_run.py"),
           "--config", str(config_file), "--output-dir", str(OUTPUT_DIR)]
print("Configuration:", config_file)
print("Output:", OUTPUT_DIR)
print("Live log:", LOG_PATH)
print(json.dumps({key: run_config[key] for key in ["run", "data", "model", "lora", "training"]}, ensure_ascii=False, indent=2))

## Preflight only — no training

Checks local data/splits, CUDA, bitsandbytes, dependencies, and source provenance without downloading/loading the model or creating the run directory.

In [ ]:
run_streamed(command + ["--preflight-only"])

## Start training

Runs the actual paired unadapted/QLoRA experiment. Live output appears below and is saved to `LOG_PATH`. Model loading and generation-based validation may take time without printing a new training step. **Interrupt this cell** to stop its process group. Do not start another run against the same output directory.

In [ ]:
print("Starting local training. Interrupt this cell to stop the launched process group.", flush=True)
print("Log file:", LOG_PATH, flush=True)
run_streamed(command, log_path=LOG_PATH)

## Completed results

Available after successful training only. Each seed retains its best adapter, epoch metrics/adapters, baseline and adapted predictions, and run manifest. ZIP packaging follows `run.archive`. These are private local artifacts, not an official competition submission.

In [ ]:
from IPython.display import JSON, FileLink, display

for filename in ["comparison.json", "source_manifest.json"]:
    print(filename)
    display(JSON(json.loads((OUTPUT_DIR / filename).read_text(encoding="utf-8"))))
for seed in run_config["run"]["seeds"]:
    print("Saved adapter:", OUTPUT_DIR / f"seed-{seed}" / "adapter")
archive_path = Path(str(OUTPUT_DIR) + "-artifacts.zip")
for path in [LOG_PATH, archive_path]:
    if path.is_file():
        print(path)
        # Relative URLs are served by Jupyter when the file is inside its workspace.
        try:
            display(FileLink(str(path.relative_to(Path.cwd()))))
        except ValueError:
            print("Outside the notebook working directory; open the file at the path above.")